# **Eksperimen MSML - Kendrick Filbert**

## **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Breast Cancer Wisconsin (Diagnostic)** dari UCI Machine Learning Repository, tersedia di scikit-learn.

**Deskripsi Dataset:**
- **Jumlah sampel:** 569
- **Jumlah fitur:** 30 fitur numerik (mean, standard error, worst dari 10 pengukuran sel)
- **Target:** Binary classification - Malignant (0) dan Benign (1)
- **Sumber:** Dr. William H. Wolberg, University of Wisconsin Hospitals, Madison

## **2. Import Library**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
print('All libraries imported successfully!')

## **3. Memuat Dataset**

In [ ]:
cancer = load_breast_cancer()
df = pd.DataFrame(data=cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df.to_csv('../breast_cancer_raw.csv', index=False)
print(f'Dataset berhasil dimuat dan disimpan sebagai CSV!')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
print('=' * 60)
print('INFORMASI DATASET')
print('=' * 60)
print(f'Jumlah sampel: {df.shape[0]}')
print(f'Jumlah fitur: {df.shape[1] - 1}')
print(f'\nTarget classes:')
print(f'  0 - Malignant (Ganas): {(df["target"] == 0).sum()}')
print(f'  1 - Benign (Jinak): {(df["target"] == 1).sum()}')
print(f'\nTipe data:')
print(df.dtypes.value_counts())

In [ ]:
df.info()

In [ ]:
df.describe().round(3)

## **4. Exploratory Data Analysis (EDA)**
### 4.1 Distribusi Target Variable

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
target_counts = df['target'].value_counts()
colors = ['#e74c3c', '#2ecc71']
axes[0].bar(['Malignant (0)', 'Benign (1)'], target_counts.values, color=colors)
axes[0].set_title('Distribusi Target Variable', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Jumlah Sampel')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')
axes[1].pie(target_counts.values, labels=['Malignant', 'Benign'],
            autopct='%1.1f%%', colors=colors, startangle=90, explode=(0.05, 0))
axes[1].set_title('Proporsi Target Variable', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Rasio Malignant:Benign = {target_counts[0]}:{target_counts[1]}')

### 4.2 Missing Values Analysis

In [ ]:
missing = df.isnull().sum()
print('Missing Values Analysis:')
print('=' * 40)
print(f'Total missing values: {missing.sum()}')
if missing.sum() == 0:
    print('Tidak ada missing values pada dataset ini!')

### 4.3 Duplicate Data Analysis

In [ ]:
duplicates = df.duplicated().sum()
print(f'Jumlah data duplikat: {duplicates}')
if duplicates == 0:
    print('Tidak ada data duplikat!')

### 4.4 Distribusi Fitur

In [ ]:
mean_features = [col for col in df.columns if 'mean' in col]
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(mean_features):
    axes[i].hist(df[df['target']==1][col], bins=30, alpha=0.5, label='Benign', color='#2ecc71')
    axes[i].hist(df[df['target']==0][col], bins=30, alpha=0.5, label='Malignant', color='#e74c3c')
    axes[i].set_title(col.replace('mean ', ''), fontsize=10, fontweight='bold')
    axes[i].legend(fontsize=7)
plt.suptitle('Distribusi Fitur Mean berdasarkan Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 4.5 Correlation Analysis

In [ ]:
plt.figure(figsize=(20, 16))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Correlation Matrix - Breast Cancer Dataset', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
target_corr = df.corr()['target'].drop('target').sort_values(ascending=False)
print('Top 10 fitur berkorelasi positif dengan target (Benign):')
print(target_corr.head(10).round(4))
print('\nTop 10 fitur berkorelasi negatif dengan target (Malignant):')
print(target_corr.tail(10).round(4))

### 4.6 Boxplot untuk Deteksi Outlier

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()
for i, col in enumerate(mean_features):
    data_to_plot = [df[df['target']==0][col], df[df['target']==1][col]]
    bp = axes[i].boxplot(data_to_plot, labels=['Malignant', 'Benign'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][1].set_facecolor('#2ecc71')
    axes[i].set_title(col.replace('mean ', ''), fontsize=10, fontweight='bold')
plt.suptitle('Boxplot Fitur Mean per Kelas Target', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
print('Analisis Outlier (IQR Method):')
print('=' * 50)
outlier_summary = []
for col in df.columns[:-1]:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    if outliers > 0:
        outlier_summary.append({'Feature': col, 'Outliers': outliers, 'Percentage': f'{outliers/len(df)*100:.1f}%'})
outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
print(outlier_df.to_string(index=False))
print(f'\nTotal fitur yang memiliki outlier: {len(outlier_df)}/30')

### 4.7 Pairplot Fitur Terpenting

In [ ]:
top5_features = target_corr.abs().sort_values(ascending=False).head(5).index.tolist()
plot_df = df[top5_features + ['target']].copy()
plot_df['target'] = plot_df['target'].map({0: 'Malignant', 1: 'Benign'})
sns.pairplot(plot_df, hue='target', palette={'Malignant': '#e74c3c', 'Benign': '#2ecc71'}, diag_kind='kde')
plt.suptitle('Pairplot - Top 5 Fitur', fontsize=16, fontweight='bold', y=1.02)
plt.show()

## **5. Data Preprocessing**
### 5.1 Menangani Duplikasi Data

In [ ]:
print(f'Shape sebelum hapus duplikasi: {df.shape}')
df_clean = df.drop_duplicates()
print(f'Shape setelah hapus duplikasi: {df_clean.shape}')
print(f'Baris yang dihapus: {df.shape[0] - df_clean.shape[0]}')

### 5.2 Menangani Missing Values

In [ ]:
print('Missing values per kolom:')
print(df_clean.isnull().sum().sum())
print('Tidak ada missing values yang perlu ditangani.')

### 5.3 Deteksi dan Penanganan Outlier (IQR Capping)

In [ ]:
feature_cols = df_clean.columns[:-1]
df_processed = df_clean.copy()
outliers_capped = 0
for col in feature_cols:
    Q1 = df_processed[col].quantile(0.25)
    Q3 = df_processed[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    before = ((df_processed[col] < lower_bound) | (df_processed[col] > upper_bound)).sum()
    outliers_capped += before
    df_processed[col] = df_processed[col].clip(lower=lower_bound, upper=upper_bound)
print(f'Total outlier yang di-capped: {outliers_capped}')
print('Outlier berhasil ditangani dengan IQR Capping!')

### 5.4 Feature-Target Split

In [ ]:
X = df_processed.drop('target', axis=1)
y = df_processed['target']
print(f'Shape X: {X.shape}')
print(f'Shape y: {y.shape}')

### 5.5 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training set: {X_train.shape[0]} sampel')
print(f'Testing set: {X_test.shape[0]} sampel')
print(f'\nDistribusi target training: {y_train.value_counts().to_dict()}')
print(f'Distribusi target testing: {y_test.value_counts().to_dict()}')

### 5.6 Standarisasi Fitur

In [ ]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
print('Standarisasi berhasil diterapkan!')
print(f'Mean fitur (train) ~0: {X_train_scaled.mean().mean():.6f}')
print(f'Std fitur (train) ~1: {X_train_scaled.std().mean():.6f}')

### 5.7 Verifikasi dan Simpan

In [ ]:
print('=' * 60)
print('RINGKASAN PREPROCESSING')
print('=' * 60)
print(f'Dataset awal: {df.shape}')
print(f'Setelah hapus duplikasi: {df_clean.shape}')
print(f'Setelah outlier capping: {df_processed.shape}')
print(f'X_train (scaled): {X_train_scaled.shape}')
print(f'X_test (scaled): {X_test_scaled.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test: {y_test.shape}')
print(f'\nData siap untuk tahap modelling!')

In [ ]:
train_data = X_train_scaled.copy()
train_data['target'] = y_train.values
test_data = X_test_scaled.copy()
test_data['target'] = y_test.values
train_data.to_csv('breast_cancer_train_preprocessing.csv', index=False)
test_data.to_csv('breast_cancer_test_preprocessing.csv', index=False)
print('Data preprocessing berhasil disimpan!')
print(f'  - breast_cancer_train_preprocessing.csv ({train_data.shape})')
print(f'  - breast_cancer_test_preprocessing.csv ({test_data.shape})')